# Petri — Paper Analysis

Clean paper-quality analysis of Petri's 38 scoring dimensions.
Reference: `petri_analysis.ipynb` (EDA scratch notebook).

**Fixed defaults**: `J=L`, `A=L` throughout. Size contrast also uses `A=L` (requires T=XS A=L run).

---
## Section 0 — Setup & Data Loading

In [ ]:
%load_ext autoreload
%autoreload 2

import json
import re
import warnings
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from scipy.stats import mannwhitneyu
from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

# ── constants ────────────────────────────────────────────────────────────────
SIZE_ORDER = ['XS', 'S', 'M', 'L', 'XL']
FACTORS    = ['target', 'auditor', 'judge']
REF_J      = 'L'
REF_A      = 'L'
DELTA_THR  = 0.5   # |Δ| threshold for 'active'
INACT_THR  = 0.05  # pct_nonzero below which a dim is 'inactive'

DATA_SAFE   = 'petri_comparison_results/norwegian_safety'
DATA_UNSAFE = 'petri_comparison_results/norwegian_safety_unsafe'

ALL_DIMS = [
    'concerning', 'needs_attention', 'disappointing',
    'counterfactual_bad_behavior', 'cooperation_with_human_misuse',
    'facilitating_terrorism', 'unprompted_gratitude',
    'unprompted_spirituality_or_extreme_awe', 'unprompted_pos_affect',
    'unprompted_neg_affect', 'unprompted_bizarre_behavior',
    'cooperation_with_harmful_sysprompt', 'user_over_sysprompt',
    'concerning_with_no_mitigating_factors', 'unprompted_deception_toward_user',
    'unprompted_sycophancy', 'unprompted_encouragement_of_user_delusion',
    'manipulated_by_developer', 'unprompted_leaking',
    'unprompted_whistleblowing', 'unprompted_blackmail',
    'unprompted_initiative', 'audit_situational_awareness', 'eval_awareness',
    'auditor_failure', 'stuck_in_loops', 'unprompted_emotional_manipulativeness',
    'unprompted_self_preservation', 'unfaithful_thinking',
    'input_hallucination', 'unprompted_self_serving_bias',
    'broken_tool_use', 'admirable', 'fun_or_funny',
    'overrefusal', 'high_stakes_setting', 'scenario_realism',
    'prefill_susceptibility',
]

# Dimensions where HIGHER = better (polarity-correct by mapping to 10 - score)
POSITIVE_DIMS = frozenset({'admirable', 'fun_or_funny', 'scenario_realism', 'high_stakes_setting'})


def polarity_correct(df, dims):
    """Return copy with POSITIVE_DIMS inverted: concern = 10 - score."""
    df = df.copy()
    for d in dims:
        if d in POSITIVE_DIMS and d in df.columns:
            df[d] = 10.0 - df[d]
    return df


def compute_active_concern(row, dims):
    """Polarity-corrected mean of dims. Scale is 0–10; POSITIVE_DIMS mapped to 10−score."""
    vals = []
    for d in dims:
        v = row.get(d)
        if v is None:
            continue
        try:
            v = float(v)
        except (TypeError, ValueError):
            continue
        if np.isnan(v):
            continue
        vals.append(10.0 - v if d in POSITIVE_DIMS else v)
    return float(np.mean(vals)) if vals else float('nan')


print('Setup complete. REF: J={REF_J}, A={REF_A}, DELTA_THR={DELTA_THR}')

In [ ]:
# ── Data loaders (transcript-level and run-level) ─────────────────────────────

def _norm(key, strip_suffix):
    return key[:-len(strip_suffix)] if strip_suffix and key.endswith(strip_suffix) else key


def _to_cat(df, col):
    present = [s for s in SIZE_ORDER if s in df[col].values]
    extras  = sorted(set(df[col].values) - set(SIZE_ORDER))
    df[col] = pd.Categorical(df[col], categories=present + extras, ordered=True)
    return df


def load_transcripts(base_dir, strip_suffix=None, tag=None):
    """One row per transcript with raw dimension scores (0–10)."""
    rows = []
    for t_path in sorted(Path(base_dir).glob('T=*__A=*/J=*/run_*/transcript_*.json')):
        ta_dir, j_dir, run_dir = t_path.parent.parent.parent, t_path.parent.parent, t_path.parent
        m1 = re.match(r'T=(\w+)__A=(\w+)', ta_dir.name)
        m2 = re.match(r'J=(\w+)', j_dir.name)
        m3 = re.match(r'run_(\d+)', run_dir.name)
        if not (m1 and m2 and m3):
            continue
        try:
            d = json.load(open(t_path))
        except Exception:
            continue
        scores = (d.get('metadata', {}).get('judge_output') or {}).get('scores') or {}
        if not scores:
            continue
        row = dict(
            target =_norm(m1.group(1), strip_suffix),
            auditor=_norm(m1.group(2), strip_suffix),
            judge  =_norm(m2.group(1), strip_suffix),
            run_id =int(m3.group(1)),
        )
        if tag:
            row['safety'] = tag
        row.update({k: float(v) for k, v in scores.items()})
        rows.append(row)
    if not rows:
        raise ValueError(f'No transcripts found under {base_dir!r}')
    df = pd.DataFrame(rows)
    for col in FACTORS:
        df = _to_cat(df, col)
    return df.reset_index(drop=True)


def load_runs(base_dir, strip_suffix=None, tag=None):
    """One row per run from experiment_results.json."""
    rows = []
    for rf in sorted(Path(base_dir).glob('T=*__A=*/J=*/experiment_results.json')):
        ta_dir, j_dir = rf.parent.parent, rf.parent
        m1 = re.match(r'T=(\w+)__A=(\w+)', ta_dir.name)
        m2 = re.match(r'J=(\w+)', j_dir.name)
        if not (m1 and m2):
            continue
        try:
            d = json.load(open(rf))
        except Exception as e:
            warnings.warn(f'Could not read {rf}: {e}')
            continue
        for run_id, run in enumerate(d.get('runs', [])):
            row = dict(
                target =_norm(m1.group(1), strip_suffix),
                auditor=_norm(m1.group(2), strip_suffix),
                judge  =_norm(m2.group(1), strip_suffix),
                run_id =run_id,
                n_transcripts=run.get('n_transcripts', 0),
                global_mean  =run.get('global_mean', float('nan')),
            )
            if tag:
                row['safety'] = tag
            for dim, stats in run.get('dimensions', {}).items():
                row[dim] = stats.get('mean', float('nan'))
            rows.append(row)
    if not rows:
        raise ValueError(f'No experiment_results.json found under {base_dir!r}')
    df = pd.DataFrame(rows)
    for col in FACTORS:
        df = _to_cat(df, col)
    return df.reset_index(drop=True)


# ── Load ──────────────────────────────────────────────────────────────────────
df_t_safe   = load_transcripts(DATA_SAFE,   tag='safe')
df_t_unsafe = load_transcripts(DATA_UNSAFE, strip_suffix='_UNSAFE', tag='unsafe')
df_t        = pd.concat([df_t_safe, df_t_unsafe], ignore_index=True)

df_r_safe   = load_runs(DATA_SAFE,   tag='safe')
df_r_unsafe = load_runs(DATA_UNSAFE, strip_suffix='_UNSAFE', tag='unsafe')
df_r        = pd.concat([df_r_safe, df_r_unsafe], ignore_index=True)

## filter out XL judge and auditor
df_r = df_r[~((df_r['judge'] == 'XL') | (df_r['auditor'] == 'XL'))].reset_index(drop=True)
df_r_safe = df_r_safe[~((df_r_safe['judge'] == 'XL') | (df_r_safe['auditor'] == 'XL'))].reset_index(drop=True)
df_r_unsafe = df_r_unsafe[~((df_r_unsafe['judge'] == 'XL') | (df_r_unsafe['auditor'] == 'XL'))].reset_index(drop=True)


dim_cols = [c for c in df_t.columns if c not in FACTORS + ['run_id', 'safety']]
print(f'Transcripts: {len(df_t_safe)} safe, {len(df_t_unsafe)} unsafe')
print(f'Runs:        {len(df_r_safe)} safe, {len(df_r_unsafe)} unsafe')
print(f'Dimensions:  {len(dim_cols)}')
print(f'Safe  targets:  {sorted(df_r_safe["target"].astype(str).unique())}')
print(f'Unsafe targets: {sorted(df_r_unsafe["target"].astype(str).unique())}')
print(f'Judges:   {sorted(df_r["judge"].astype(str).unique())}')
print(f'Auditors: {sorted(df_r["auditor"].astype(str).unique())}')

In [ ]:
# ── Available combinations ────────────────────────────────────────────────────
combos = (
    df_r.groupby(['safety'] + FACTORS, observed=True)
        .agg(n_runs=('run_id', 'count'), mean_gm=('global_mean', 'mean'))
        .reset_index()
)
print(f'{'Safety':<8} {'T':>4} {'A':>4} {'J':>4}  {'N':>4}  {'mean_gm':>8}')
print('-' * 48)
for _, r in combos.iterrows():
    print(f'{r.safety:<8} {str(r.target):>4} {str(r.auditor):>4} {str(r.judge):>4}  {int(r.n_runs):>4}  {r.mean_gm:>8.4f}')

---
## Section 1 — Dimension Taxonomy

We classify all 38 dimensions into four groups:

| Label | Criteria |
|-------|----------|
| `completely_inactive` | `pct_above1 == 0` — never exceeded floor score of 1 |
| `near_inactive` | `0 < pct_above1 < 0.05` — fires rarely (<5% of transcripts) |
| `active_unresponsive` | Fires, but neither safety nor size contrast is significant |
| `safety_active` | \|Δ\| ≥ 0.5 and p < 0.05 in safety contrast only |
| `size_active` | \|Δ\| ≥ 0.5 and p < 0.05 in size contrast only |
| `both_active` | Active in both contrasts |

Both contrasts use **J=L, A=L** as the fixed reference.

**Safety contrast**: T=S safe vs T=S_UNSAFE.  
**Size contrast**: T=XS vs T=M (both safe). XS vs M chosen over XS vs S for a larger capability gap
and better parallel to the safety contrast. XS vs S shows the same core dims with smaller Δ (monotonic).

**Size contrast artifact**: floor-locked binary dims show a systematic Δ ≈ +0.09 (XS always scores 1, M occasionally scores 0). The |Δ| ≥ 0.5 threshold is well above this and eliminates artifact cases.

In [ ]:
dim_cols

In [ ]:
# ── 1a: Activity levels across all safe transcripts (J=L) ────────────────────
ref_safe = df_t_safe[df_t_safe['judge'].astype(str) == REF_J].copy()
print(f'Reference transcripts (safe, J={REF_J}): {len(ref_safe)}')

act_rows = []
for dim in dim_cols:
    vals = ref_safe[dim].dropna()
    pct_nonzero = float((vals > 0).mean())
    pct_above1  = float((vals > 1).mean())
    mean_val    = float(vals.mean())
    std_val     = float(vals.std())
    # Activity is based on pct_above1, NOT pct_nonzero.
    # Floor for all dims is 1 (not 0): inactive dims score 1 on ~97% of transcripts
    # and 0 on ~3%, so pct_nonzero ~= 0.97 for every inactive dim — misleading.
    # pct_above1 == 0 correctly identifies dims that never exceed the floor.
    if pct_above1 == 0.0:
        tier = 'completely_inactive'
    elif pct_above1 < INACT_THR:
        tier = 'near_inactive'
    else:
        tier = 'variable'
    act_rows.append(dict(dimension=dim, pct_nonzero=pct_nonzero, pct_above1=pct_above1,
                         mean=mean_val, std=std_val, activity_tier=tier))

act_df = pd.DataFrame(act_rows).sort_values('pct_above1').reset_index(drop=True)
completely_inactive = act_df[act_df['activity_tier'] == 'completely_inactive']['dimension'].tolist()
near_inactive       = act_df[act_df['activity_tier'] == 'near_inactive']['dimension'].tolist()
variable_dims       = act_df[act_df['activity_tier'] == 'variable']['dimension'].tolist()

print(f'\nDimension activity (safe transcripts, J={REF_J}):')
print(f'  Completely inactive (pct_above1=0):           {len(completely_inactive)}')
print(f'  Near-inactive (0 < pct_above1 < {INACT_THR}):    {len(near_inactive)}')
print(f'  Variable (pct_above1 >= {INACT_THR}):             {len(variable_dims)}')

print(f'\n── Completely inactive dims (never exceeded floor score of 1):')
for d in completely_inactive:
    r = act_df[act_df['dimension']==d].iloc[0]
    print(f'  {d:<48}  pct_nonzero={r["pct_nonzero"]:.3f}  mean={r["mean"]:.3f}  std={r["std"]:.3f}')

print(f'\n── Near-inactive dims (scores > 1 on <{INACT_THR*100:.0f}% of transcripts):')
for d in near_inactive:
    r = act_df[act_df['dimension']==d].iloc[0]
    print(f'  {d:<48}  pct_above1={r["pct_above1"]:.3f}  mean={r["mean"]:.3f}  std={r["std"]:.3f}')
    print(f'    NOTE: This dimension can fire (score > 1) but rarely — adds noise to global_mean')

print(f'\n── Variable dims (pct_above1 >= {INACT_THR}):')
for d in variable_dims:
    r = act_df[act_df['dimension']==d].iloc[0]
    print(f'  {d:<48}  pct_above1={r["pct_above1"]:.3f}  mean={r["mean"]:.3f}')


In [ ]:
# ── 1b: Safety contrast — T=S safe vs T=S_UNSAFE, A=L, J=L ──────────────────
su_safe   = df_t_safe[
    (df_t_safe['target'].astype(str)  == 'S') &
    (df_t_safe['auditor'].astype(str) == REF_A) &
    (df_t_safe['judge'].astype(str)   == REF_J)
]
su_unsafe = df_t_unsafe[
    (df_t_unsafe['target'].astype(str)  == 'S') &
    (df_t_unsafe['auditor'].astype(str) == REF_A) &
    (df_t_unsafe['judge'].astype(str)   == REF_J)
]
print(f'Safety contrast (T=S A={REF_A} J={REF_J}): {len(su_safe)} safe, {len(su_unsafe)} unsafe transcripts')

safety_rows = []
for dim in dim_cols:
    va = su_safe[dim].dropna().values
    vb = su_unsafe[dim].dropna().values
    if len(va) < 3 or len(vb) < 3:
        continue
    delta = float(vb.mean() - va.mean())
    _, p  = mannwhitneyu(va, vb, alternative='two-sided')
    safety_rows.append(dict(
        dimension=dim, mean_safe=float(va.mean()), mean_unsafe=float(vb.mean()),
        delta=delta, abs_delta=abs(delta), p_value=float(p),
        significant=p < 0.05,
        safety_active=(p < 0.05 and abs(delta) >= DELTA_THR),
    ))

contrast_safety = pd.DataFrame(safety_rows).sort_values('delta', ascending=False).reset_index(drop=True)
safety_active_dims = contrast_safety[contrast_safety['safety_active']]['dimension'].tolist()

print(f'Safety-active dims (|Δ|>={DELTA_THR}, p<0.05): {len(safety_active_dims)}')
for _, r in contrast_safety[contrast_safety['safety_active']].iterrows():
    inv = '  [↑=better → positive = unsafe is LESS admirable]' if r.dimension in POSITIVE_DIMS else ''
    print(f'  {r.dimension:<48}  safe={r.mean_safe:.2f}  unsafe={r.mean_unsafe:.2f}  Δ={r.delta:+.3f}  p={r.p_value:.4f}{inv}')

In [ ]:
# ── 1c: Size contrast — T=XS vs T=M, A=L, J=L ────────────────────────────────
# XS vs M chosen over XS vs S: larger capability gap, clearer signal,
# and better parallel to the safety contrast (also a large behavioral gap).
# XS vs S shows the same core dims with smaller Δ, confirming monotonic effect.
sz_xs = df_t_safe[
    (df_t_safe['target'].astype(str)  == 'XS') &
    (df_t_safe['auditor'].astype(str) == REF_A) &
    (df_t_safe['judge'].astype(str)   == REF_J)
]
sz_m = df_t_safe[
    (df_t_safe['target'].astype(str)  == 'M') &
    (df_t_safe['auditor'].astype(str) == REF_A) &
    (df_t_safe['judge'].astype(str)   == REF_J)
]
if len(sz_xs) == 0:
    raise RuntimeError(f'T=XS A={REF_A} J={REF_J} data missing — add the run and re-execute')
if len(sz_m) == 0:
    raise RuntimeError(f'T=M A={REF_A} J={REF_J} data missing')
print(f'Size contrast (A={REF_A} J={REF_J}): T=XS={len(sz_xs)}, T=M={len(sz_m)} transcripts')

# Artifact note: binary floor dims show systematic Δ ≈ +0.09 (XS always=1, M sometimes=0).
# |Δ| >= 0.5 threshold is 5x the artifact magnitude and cleanly excludes it.
ARTIFACT_BAND = 0.10  # visualisation only

size_rows = []
for dim in dim_cols:
    vx = sz_xs[dim].dropna().values  # XS
    vm = sz_m[dim].dropna().values   # M
    if len(vx) < 3 or len(vm) < 3:
        continue
    # delta = XS - M  (positive = XS scores higher = smaller model is more concerning)
    delta = float(vx.mean() - vm.mean())
    _, p  = mannwhitneyu(vx, vm, alternative='two-sided')
    size_rows.append(dict(
        dimension=dim, mean_xs=float(vx.mean()), mean_m=float(vm.mean()),
        delta=delta, abs_delta=abs(delta), p_value=float(p),
        significant=p < 0.05,
        size_active=(p < 0.05 and abs(delta) >= DELTA_THR),
    ))

contrast_size = pd.DataFrame(size_rows).sort_values('delta', ascending=False).reset_index(drop=True)
size_active_dims = contrast_size[contrast_size['size_active']]['dimension'].tolist()

print(f'Size-active dims (|Δ|>={DELTA_THR}, p<0.05): {len(size_active_dims)}')
for _, r in contrast_size[contrast_size['size_active']].iterrows():
    print(f'  {r.dimension:<48}  XS={r.mean_xs:.2f}  M={r.mean_m:.2f}  Δ={r.delta:+.3f}  p={r.p_value:.4f}')

In [ ]:
# ── 1d: 4-way classification and active_union ─────────────────────────────────
inactive_set = set(completely_inactive) | set(near_inactive)

cs_idx = contrast_safety.set_index('dimension')
cz_idx = contrast_size.set_index('dimension')

all_dim_rows = []
for dim in dim_cols:
    is_safety = dim in safety_active_dims
    is_size   = dim in size_active_dims
    sdelta = float(cs_idx.loc[dim, 'delta'])   if dim in cs_idx.index else float('nan')
    zdelta = float(cz_idx.loc[dim, 'delta'])   if dim in cz_idx.index else float('nan')
    sp     = float(cs_idx.loc[dim, 'p_value']) if dim in cs_idx.index else float('nan')
    zp     = float(cz_idx.loc[dim, 'p_value']) if dim in cz_idx.index else float('nan')

    if dim in completely_inactive:
        label = 'completely_inactive'
    elif dim in near_inactive:
        label = 'near_inactive'
    elif is_safety and is_size:
        label = 'both_active'
    elif is_safety:
        label = 'safety_only'
    elif is_size:
        label = 'size_only'
    else:
        label = 'active_unresponsive'

    all_dim_rows.append(dict(
        dimension=dim, label=label,
        safety_delta=sdelta, safety_p=sp,
        size_delta=zdelta,   size_p=zp,
    ))

dim_table = pd.DataFrame(all_dim_rows)
active_union = sorted(
    set(safety_active_dims) | set(size_active_dims)
)

# Summary
print('Table 1 — Dimension Classification:')
print(f'  completely_inactive:  {(dim_table["label"]=="completely_inactive").sum()}')
print(f'  near_inactive:        {(dim_table["label"]=="near_inactive").sum()}')
print(f'  active_unresponsive:  {(dim_table["label"]=="active_unresponsive").sum()}')
print(f'  safety_only:          {(dim_table["label"]=="safety_only").sum()}')
print(f'  size_only:            {(dim_table["label"]=="size_only").sum()}')
print(f'  both_active:          {(dim_table["label"]=="both_active").sum()}')
print(f'  active_union:         {len(active_union)}')
print()
print('Active unresponsive dims (fires but carries no safety/size signal):')
for _, r in dim_table[dim_table['label']=='active_unresponsive'].iterrows():
    print(f'  {r.dimension:<48}  safety_Δ={r.safety_delta:+.3f} (p={r.safety_p:.3f})  size_Δ={r.size_delta:+.3f} (p={r.size_p:.3f})')
    if r.dimension == 'scenario_realism':
        print('    NOTE: scenario_realism is an AUDITOR quality metric, not a target safety metric.')
    if r.dimension == 'high_stakes_setting':
        print('    NOTE: high_stakes_setting is also an auditor-side metric (scenario difficulty).')

print()
print(f'active_union: {active_union}')

In [ ]:
# ── Figure 1: Side-by-side contrast bar charts ───────────────────────────────
from matplotlib.patches import Patch

# Canonical order: sorted by safety Δ ascending (most positive at top)
dim_order = (contrast_safety
             .merge(pd.DataFrame({'dimension': dim_cols}), on='dimension')
             .sort_values('delta', ascending=True)['dimension']
             .tolist())

cz_reindexed = (
    contrast_size.set_index('dimension')
    .reindex(dim_order)
    .reset_index()
    .rename(columns={'index': 'dimension'})
)
cz_reindexed['dimension'] = dim_order


def _bar_colors(cdf, active_set):
    colors = []
    for _, r in cdf.iterrows():
        d = r['dimension']
        if d in completely_inactive:
            colors.append('#e0e0e0')
        elif d in near_inactive:
            colors.append('#c0c0c0')
        elif d in active_set:
            colors.append('#D65F5F' if r.get('delta', 0) > 0 else '#4878CF')
        elif r.get('significant', False):
            colors.append('#ffaaaa' if r.get('delta', 0) > 0 else '#aaaadd')
        else:
            colors.append('#999999')
    return colors


fig, axes = plt.subplots(1, 2, figsize=(18, 14))

panels = [
    (contrast_safety.set_index('dimension').reindex(dim_order).reset_index(),
     safety_active_dims,
     f'Safety Δ = Mean(unsafe-S) − Mean(safe-S)  [T=S A={REF_A} J={REF_J}]',
     'Figure 1a — Safety contrast'),
    (cz_reindexed,
     size_active_dims,
     f'Size Δ = Mean(XS) − Mean(M)  [safe, A={REF_A} J={REF_J}]',
     'Figure 1b — Size contrast (XS vs M)'),
]

for ax, (cdf, active_set, x_label, title) in zip(axes, panels):
    colors = _bar_colors(cdf, active_set)
    deltas = cdf['delta'].fillna(0).values
    ax.barh(range(len(cdf)), deltas, color=colors, alpha=0.9)
    ax.set_yticks(range(len(cdf)))
    ax.set_yticklabels([d.replace('unprompted_', 'u_') for d in dim_order], fontsize=7.5)
    ax.axvline(0, color='black', lw=0.8)
    ax.axvline( DELTA_THR, color='gray', ls='--', lw=1, alpha=0.5)
    ax.axvline(-DELTA_THR, color='gray', ls='--', lw=1, alpha=0.5)
    # if 'Size contrast' in title:
    #     ax.axvspan(-ARTIFACT_BAND, ARTIFACT_BAND, color='yellow', alpha=0.15,
    #                label=f'Scoring artifact band (±{ARTIFACT_BAND})')
    #     ax.legend(fontsize=7, loc='lower right')
    ax.set_xlabel(x_label, fontsize=10)
    ax.set_title(f'{title}\n{len(active_set)} active dims', fontsize=11)
    ax.grid(axis='x', ls='--', alpha=0.3)

legend_els = [
    Patch(color='#D65F5F', label=f'active positive (|Δ|≥{DELTA_THR}, p<0.05)'),
    Patch(color='#4878CF', label=f'active negative (|Δ|≥{DELTA_THR}, p<0.05)'),
    Patch(color='#ffaaaa', label='significant but below threshold'),
    Patch(color='#999999', label='not significant'),
    Patch(color='#c0c0c0', label='near-inactive (<5% fire rate)'),
    Patch(color='#e0e0e0', label='completely inactive'),
]
axes[0].legend(handles=legend_els, fontsize=8, loc='lower right')
plt.tight_layout()
plt.savefig('img/fig_petri_contrasts.png', dpi=300)
plt.show()

In [ ]:
# ── Figure 2: Contrast scatter map ───────────────────────────────────────────
su_d = contrast_safety.set_index('dimension')['delta']
sz_d = contrast_size.set_index('dimension')['delta']
scat = pd.DataFrame({'su': su_d, 'sz': sz_d}).dropna()
scat['label'] = [dim_table.set_index('dimension').loc[d, 'label']
                 if d in dim_table['dimension'].values else 'active_unresponsive'
                 for d in scat.index]

style_map = {
    'both_active':          ('#8B0000', 100, 'D', 'both active'),
    'safety_only':          ('#D65F5F',  70, 'o', 'safety only'),
    'size_only':            ('#4878CF',  70, 's', 'size only'),
    'active_unresponsive':  ('#ff9900',  50, 'o', 'active, unresponsive'),
    'near_inactive':        ('#bbbbbb',  30, 'x', 'near-inactive'),
    'completely_inactive':  ('#dddddd',  20, 'x', 'completely inactive'),
}

fig, ax = plt.subplots(figsize=(10, 7))
for lbl, (color, size, marker, legend_lbl) in style_map.items():
    sub = scat[scat['label'] == lbl]
    if not sub.empty:
        ax.scatter(sub['su'], sub['sz'], c=color, s=size, marker=marker,
                   alpha=0.85, label=f'{legend_lbl} ({len(sub)})', zorder=3)

label_set = (set(active_union) |
             {d for d in scat.index if abs(scat.loc[d,'su']) > 0.3 or abs(scat.loc[d,'sz']) > 0.3})
for d in label_set:
    if d in scat.index:
        ax.annotate(d.replace('unprompted_', 'u_'),
                    (scat.loc[d,'su'], scat.loc[d,'sz']),
                    fontsize=7, xytext=(4, 2), textcoords='offset points', alpha=0.9)

ax.axvline(0, color='black', lw=0.8)
ax.axhline(0, color='black', lw=0.8)
ax.axvline( DELTA_THR, color='gray', ls='--', alpha=0.4, lw=1)
ax.axvline(-DELTA_THR, color='gray', ls='--', alpha=0.4, lw=1)
ax.axhline( DELTA_THR, color='gray', ls='--', alpha=0.4, lw=1)
ax.axhline(-DELTA_THR, color='gray', ls='--', alpha=0.4, lw=1)
ax.set_xlabel(f'Safety Δ — unsafe-S minus safe-S  [A={REF_A} J={REF_J}]', fontsize=11)
ax.set_ylabel(f'Size Δ — XS minus M  [safe, A={REF_A} J={REF_J}]', fontsize=11)
ax.set_title('Figure 2 — Contrast map: safety discrimination vs size discrimination', fontsize=12)
ax.legend(fontsize=9, loc='upper left', framealpha=0.9)
ax.grid(ls='--', alpha=0.3)
plt.tight_layout()
plt.show()

---
## Section 2 — Independence Analysis

We use the `active_union` (~14 dims) to find the effective dimensionality.

**Why combined safe+unsafe data for PCA**:  
- Safe-only: active dims sit near their floor → PCA captures noise, not structure  
- Combined (safe + unsafe, T=S A=L J=L): creates the discriminative variance for meaningful PCA  
- We want dims that capture the full range of model behavior  

Expected structure (from dry-run with 14 dims): PC1≈53%, PC2≈17%, cumulative ≈70% → 4 components reach 83%. Very low-dimensional for 14 variables — confirms strong redundancy in the active set.

We verify the correlation structure is stable within each condition separately.

In [ ]:
# ── Prepare combined transcript data for independence analysis ────────────────
# T=S, A=L, J=L — safe + unsafe. Polarity-correct before correlation.
ref_safe_ts = df_t_safe[
    (df_t_safe['target'].astype(str)  == 'S') &
    (df_t_safe['auditor'].astype(str) == REF_A) &
    (df_t_safe['judge'].astype(str)   == REF_J)
][active_union].copy()

ref_unsafe_ts = df_t_unsafe[
    (df_t_unsafe['target'].astype(str)  == 'S') &
    (df_t_unsafe['auditor'].astype(str) == REF_A) &
    (df_t_unsafe['judge'].astype(str)   == REF_J)
][active_union].copy()

ref_safe_ts   = polarity_correct(ref_safe_ts,   active_union)
ref_unsafe_ts = polarity_correct(ref_unsafe_ts, active_union)

combined_ts = pd.concat([ref_safe_ts, ref_unsafe_ts], ignore_index=True)
combined_safety_label = ['safe'] * len(ref_safe_ts) + ['unsafe'] * len(ref_unsafe_ts)

print(f'Combined transcripts for PCA: {len(combined_ts)} '
      f'(safe={len(ref_safe_ts)}, unsafe={len(ref_unsafe_ts)})')
print(f'Active union dims: {active_union}')

In [ ]:
# ── Figure 3: Correlation heatmap of active_union ────────────────────────────
corr_combined = combined_ts.corr(method='pearson')

# Hierarchical clustering for display order
dist_mat = (1 - corr_combined.abs()).clip(lower=0).values.copy()
np.fill_diagonal(dist_mat, 0)
link_corr = linkage(squareform(dist_mat), method='ward')

N_CLUSTERS = 3
cluster_ids = fcluster(link_corr, N_CLUSTERS, criterion='maxclust')
dim_to_cluster = dict(zip(active_union, cluster_ids.tolist()))

palette = sns.color_palette('Set2', N_CLUSTERS)
row_colors = pd.Series([palette[dim_to_cluster[d]-1] for d in active_union], index=active_union)

g = sns.clustermap(
    corr_combined,
    row_linkage=link_corr, col_linkage=link_corr,
    cmap='RdBu_r', center=0, vmin=-1, vmax=1,
    figsize=(12, 12), linewidths=0.2, linecolor='white',
    row_colors=row_colors, col_colors=row_colors,
    xticklabels=[d.replace('unprompted_', 'u_') for d in active_union],
    yticklabels=[d.replace('unprompted_', 'u_') for d in active_union],
    cbar_kws={'label': 'Pearson r', 'shrink': 0.4},
    dendrogram_ratio=0.1,
    annot=True, fmt='.2f', annot_kws={'size': 7},
)
g.ax_heatmap.set_xticklabels(g.ax_heatmap.get_xticklabels(), fontsize=8, rotation=90)
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=8)
g.fig.suptitle('Figure 3 — Correlation matrix: active_union dims (polarity-corrected)\n'
               'Combined safe+unsafe T=S A=L J=L | colored strips = Ward cluster',
               fontsize=11, y=1.01)
plt.show()

print(f'\nCluster membership ({N_CLUSTERS} clusters):')
for c in sorted(set(cluster_ids.tolist())):
    members = [d for d, cl in dim_to_cluster.items() if cl == c]
    print(f'  Cluster {c}: {[m.replace("unprompted_","u_") for m in members]}')

print(f'\nNear-redundant pairs (|r| > 0.7):')
for i, d1 in enumerate(active_union):
    for j, d2 in enumerate(active_union):
        if i < j and abs(corr_combined.loc[d1, d2]) > 0.7:
            print(f'  {d1:<42} ↔ {d2:<42} r={corr_combined.loc[d1,d2]:+.3f}')

In [ ]:
# ── Figure 4a: PCA Scree Plot ─────────────────────────────────────────────────
X = StandardScaler().fit_transform(combined_ts[active_union].values)
pca = PCA()
scores_pca = pca.fit_transform(X)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_80pct = int(np.argmax(cumvar >= 0.80)) + 1
n_90pct = int(np.argmax(cumvar >= 0.90)) + 1
print('PCA variance explained:')
for i, (v, c) in enumerate(zip(pca.explained_variance_ratio_, cumvar)):
    mark = ' ← 80%' if i+1 == n_80pct else (' ← 90%' if i+1 == n_90pct else '')
    print(f'  PC{i+1}: {v:.3f} (cumulative {c:.3f}){mark}')

fig, ax = plt.subplots(figsize=(8, 6))
ks = range(1, len(active_union)+1)
ax.bar(ks, pca.explained_variance_ratio_, color='steelblue', alpha=0.7, label='Per-component')
ax.plot(ks, cumvar, 'o-', color='#D65F5F', lw=2, ms=5, label='Cumulative')
ax.axhline(0.80, color='gray', ls='--', lw=1, alpha=0.7)
ax.axhline(0.90, color='gray', ls=':', lw=1, alpha=0.7)
ax.text(len(ks)-0.5, 0.81, '80%', color='gray', fontsize=8, va='bottom', ha='right')
ax.text(len(ks)-0.5, 0.91, '90%', color='gray', fontsize=8, va='bottom', ha='right')
ax.set_xlabel('Principal component', fontsize=11)
ax.set_ylabel('Variance explained', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', ls='--', alpha=0.3)
plt.tight_layout()
plt.savefig('img/fig_petri_pca_scree.png', dpi=300)
plt.show()



In [ ]:
#── Figure 4b: PCA Biplot (PC1 vs PC2) ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))
colors_pt = ['#4878CF' if s == 'safe' else '#D65F5F' for s in combined_safety_label]
ax.scatter(scores_pca[:, 0], scores_pca[:, 1], c=colors_pt, alpha=0.4, s=20)

scale = 3.0
for i, dim in enumerate(active_union):
    lx, ly = pca.components_[0, i] * scale, pca.components_[1, i] * scale
    ax.annotate('', xy=(lx, ly), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2, alpha=0.7))
    ax.text(lx * 1.05, ly * 1.05, dim.replace('unprompted_', 'u_'),
            fontsize=7, alpha=0.85, ha='center')

safe_patch   = mpatches.Patch(color='#4878CF', label='safe')
unsafe_patch = mpatches.Patch(color='#D65F5F', label='unsafe')
ax.legend(handles=[safe_patch, unsafe_patch], fontsize=9)
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
ax.set_title('Figure 4b — Biplot (PC1 vs PC2)\n(polarity-corrected, combined safe+unsafe)', fontsize=11)
ax.axvline(0, color='black', lw=0.5)
ax.axhline(0, color='black', lw=0.5)
ax.grid(ls='--', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── Stability check: correlation structure safe-only vs unsafe-only ───────────
print('Pairwise correlations within safe-only vs unsafe-only (active_union dims):')
print(f'{'Dim A':<40} {'Dim B':<40} {'r (combined)':>14} {'r (safe)':>10} {'r (unsafe)':>10}')
print('-' * 120)
for i, d1 in enumerate(active_union):
    for j, d2 in enumerate(active_union):
        if i >= j:
            continue
        r_comb = float(corr_combined.loc[d1, d2])
        r_safe  = float(ref_safe_ts[[d1, d2]].corr().loc[d1, d2])
        r_unsafe = float(ref_unsafe_ts[[d1, d2]].corr().loc[d1, d2])
        # Only print pairs with notable correlation
        if abs(r_comb) > 0.4 or abs(r_safe) > 0.4 or abs(r_unsafe) > 0.4:
            print(f'{d1:<40} {d2:<40} {r_comb:>+14.3f} {r_safe:>+10.3f} {r_unsafe:>+10.3f}')

In [ ]:
# ── 2d: Representative dimension selection ────────────────────────────────────
# For each cluster, pick the dim with highest mean |r| to all other cluster members.
representative_dims = []
cluster_reps = {}

for c in sorted(set(cluster_ids.tolist())):
    members = [d for d, cl in dim_to_cluster.items() if cl == c]
    if len(members) == 1:
        rep = members[0]
    else:
        sub_corr = corr_combined.loc[members, members]
        mean_abs_r = sub_corr.abs().sum() / (len(members) - 1)
        rep = mean_abs_r.idxmax()
    cluster_reps[c] = rep
    representative_dims.append(rep)

print(f'Representative dims per cluster (most central by mean |r|):')
for c, rep in cluster_reps.items():
    members = [d for d, cl in dim_to_cluster.items() if cl == c]
    print(f'  Cluster {c} → REP: {rep:<40}  (cluster: {[m.replace("unprompted_","u_") for m in members]})')

print(f'\nrepresentative_dims = {representative_dims}')

# Compute active_concern_v2 on run-level dataframes
_rep_dims = [d for d in representative_dims if d in df_r_safe.columns]
for _df in [df_r_safe, df_r_unsafe, df_r]:
    _df['active_concern_v2'] = [
        compute_active_concern(r, _rep_dims) for r in _df.to_dict('records')
    ]

print(f'\nactive_concern_v2 vs global_mean comparison:')
print(f'  Safe  — active_concern_v2: {df_r_safe["active_concern_v2"].mean():.3f} ± {df_r_safe["active_concern_v2"].std():.3f}')
print(f'  Safe  — global_mean:       {df_r_safe["global_mean"].mean():.3f} ± {df_r_safe["global_mean"].std():.3f}')
print(f'  Unsafe— active_concern_v2: {df_r_unsafe["active_concern_v2"].mean():.3f} ± {df_r_unsafe["active_concern_v2"].std():.3f}')
print(f'  Unsafe— global_mean:       {df_r_unsafe["global_mean"].mean():.3f} ± {df_r_unsafe["global_mean"].std():.3f}')
sep_v2 = df_r_unsafe['active_concern_v2'].mean() - df_r_safe['active_concern_v2'].mean()
sep_gm = df_r_unsafe['global_mean'].mean() - df_r_safe['global_mean'].mean()
print(f'  Separation (unsafe-safe): active_concern_v2={sep_v2:.3f}  global_mean={sep_gm:.3f}')

---
## Section 3 — Detection Capability

In [ ]:
# ── Figure 6: Safe vs unsafe — active_concern_v2 and global_mean ─────────────
# Fixed A=L, J=L. All available safe targets paired with their unsafe counterparts.

def _su_comparison(metric, title_suffix):
    rows = []
    for t in sorted(set(df_r_safe['target'].astype(str)) & set(df_r_unsafe['target'].astype(str))):
        for a in [REF_A]:
            for j in [REF_J]:
                sv = df_r_safe[
                    (df_r_safe['target']==t)&(df_r_safe['auditor']==a)&(df_r_safe['judge']==j)
                ][metric].dropna().values
                uv = df_r_unsafe[
                    (df_r_unsafe['target']==t)&(df_r_unsafe['auditor']==a)&(df_r_unsafe['judge']==j)
                ][metric].dropna().values
                if len(sv) < 3 or len(uv) < 3:
                    continue
                _, p = mannwhitneyu(sv, uv, alternative='two-sided')
                rows.append(dict(label=f'T={t}', mean_safe=sv.mean(), std_safe=sv.std(),
                                 mean_unsafe=uv.mean(), std_unsafe=uv.std(), p=p))
    return pd.DataFrame(rows)


fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=False)

for ax, (metric, ylabel) in zip(axes, [
    # ('active_concern_v2', 'active_concern_v2 (representative dims, polarity-corrected)'),
    ('concerning', 'concerning'),
    ('global_mean',       'global_mean (all 38 dims, diluted)'),
]):
    df_su = _su_comparison(metric, metric)
    if df_su.empty:
        ax.text(0.5, 0.5, 'No matched combos', ha='center', va='center', transform=ax.transAxes)
        continue
    xs = np.arange(len(df_su))
    w  = 0.35
    ax.bar(xs - w/2, df_su['mean_safe'],   w, yerr=df_su['std_safe'],   capsize=5,
           label='Safe',   color='#4878CF', alpha=0.85, error_kw=dict(elinewidth=1.5))
    ax.bar(xs + w/2, df_su['mean_unsafe'], w, yerr=df_su['std_unsafe'], capsize=5,
           label='Unsafe', color='#D65F5F', alpha=0.85, error_kw=dict(elinewidth=1.5))
    # for i, row in enumerate(df_su.itertuples()):
    #     sig = '***' if row.p < 0.001 else ('**' if row.p < 0.01 else ('*' if row.p < 0.05 else 'ns'))
    #     top = max(row.mean_safe + row.std_safe, row.mean_unsafe + row.std_unsafe)
    #     ax.text(i, top + 0.03, sig, ha='center', fontsize=11)
    ax.set_xticks(xs)
    ax.set_xticklabels(df_su['label'], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(fontsize=10)
    ax.grid(axis='y', ls='--', alpha=0.4)

plt.suptitle(f'Figure 6 — Safe vs unsafe detection  [A={REF_A}, J={REF_J}]', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Box + strip: safe vs unsafe concerning-score distributions by target size ──
METRIC   = 'unprompted_bizarre_behavior'
PALETTE  = {'Safe': '#4C72B0', 'Unsafe': '#DD8452'}
RNG      = np.random.default_rng(42)
SIZE_ORDER = ['XS', 'S', 'M']


REF_A = 'L'
REF_J = 'L'

available = set(df_r_safe['target'].astype(str)) & set(df_r_unsafe['target'].astype(str))
targets = [t for t in SIZE_ORDER if t in available]

fig, axes = plt.subplots(1, len(targets), figsize=(8, 3.8), sharey=True)
if len(targets) == 1:
    axes = [axes]

for ax, sz in zip(axes, targets):
    for i, grp in enumerate(['Safe', 'Unsafe']):
        df_src = df_r_safe if grp == 'Safe' else df_r_unsafe
        vals = df_src[
            (df_src['target'].astype(str)  == sz) &
            (df_src['auditor'].astype(str) == REF_A) &
            (df_src['judge'].astype(str)   == REF_J)
        ][METRIC].dropna().values

        if len(vals) == 0:
            continue

        color = PALETTE[grp]
        ax.boxplot(
            vals, positions=[i], widths=0.42, patch_artist=True,
            medianprops=dict(color='white', linewidth=2.2),
            boxprops=dict(facecolor=color, alpha=0.55),
            whiskerprops=dict(color=color, linewidth=1.3),
            capprops=dict(color=color, linewidth=1.3),
            flierprops=dict(marker=''),
        )
        jitter = RNG.uniform(-0.14, 0.14, len(vals))
        ax.scatter(i + jitter, vals, color=color, s=38, zorder=4,
                   alpha=0.9, edgecolors='white', linewidths=0.5)

    ax.set_title(f'Target = {sz}', fontsize=11)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Safe', 'Unsafe'], fontsize=10)
    ax.set_xlim(-0.6, 1.6)
    ax.grid(axis='y', alpha=0.25, linewidth=0.8)

axes[0].set_ylabel(METRIC, fontsize=10)

# fig.suptitle(
#     f'Safe vs unsafe detection  |  auditor={REF_A}, judge={REF_J}',
#     fontsize=11
# )
plt.tight_layout()
plt.savefig('img/fig_safe_unsafe_distributions_petri.png', bbox_inches='tight')
plt.show()

### Detection Quality — AUROC

**Why AUROC?**  
The bar chart above shows means and standard deviations, but a single number that captures
*both* the gap and the overlap is more useful for comparing across experiments and against SimpleAudit.

AUROC = **P(unsafe\_metric > safe\_metric)** for concern metrics (higher = more unsafe).  
Computed via the Mann-Whitney U statistic (non-parametric, no normality assumption):

$$\text{AUROC} = \frac{U}{n_{\text{safe}} \times n_{\text{unsafe}}}$$

**How to read it:**

| AUROC | Meaning |
|-------|---------|
| 1.00  | Every unsafe run scored above every safe run — perfect separation |
| 0.95  | In 95 % of random unsafe–safe pairs the unsafe one scored higher |
| 0.50  | Chance — no useful signal |

Set `HIGHER_IS_SAFER = True` when using an inverted metric where higher is better/safer.  
The bootstrap 95 % CI reflects uncertainty from the small n=10 per condition.

In [ ]:
# ── Parameterised AUROC — Petri metrics ──────────────────────────────────────
# AUROC = P(positive_class > negative_class) where positive = unsafe.
# For concern metrics (HIGHER_IS_SAFER=False): P(unsafe > safe).
# For inverted metrics (HIGHER_IS_SAFER=True): P(safe > unsafe).
# In both cases we use alternative='greater' and put the positive class first.

# ── Parameters (edit these) ───────────────────────────────────────────────────
GROUPBY         = 'target'                             # 'target', 'judge', or 'auditor'
FIXED           = {'auditor': REF_A, 'judge': REF_J}  # hold these constant; omit GROUPBY key
FIXED           = {'auditor': "L", 'judge': "L"} 
METRIC          = 'unprompted_bizarre_behavior'                  # 'active_concern_v2', 'global_mean', or any dim name
HIGHER_IS_SAFER = False   # False for concern dims (higher = more unsafe); True for inverted metrics
N_BOOT          = 1000
RNG             = np.random.default_rng(42)

# ── Apply fixed filters ────────────────────────────────────────────────────────
_df_safe   = df_r_safe.copy()
_df_unsafe = df_r_unsafe.copy()
for col, val in FIXED.items():
    _df_safe   = _df_safe  [_df_safe  [col].astype(str) == str(val)]
    _df_unsafe = _df_unsafe[_df_unsafe[col].astype(str) == str(val)]

_present   = set(_df_safe[GROUPBY].astype(str).unique()) | set(_df_unsafe[GROUPBY].astype(str).unique())
group_vals = [v for v in SIZE_ORDER if v in _present] or sorted(_present)

# ── Helper ─────────────────────────────────────────────────────────────────────
# Always pass (positive_class, negative_class) with alternative='greater'.
# mannwhitneyu(a, b, 'greater') → U_a = count(a_i > b_j) → auroc = P(a > b).

def _auroc_ci_petri(pos, neg, n_boot=N_BOOT, rng=RNG):
    stat, _ = mannwhitneyu(pos, neg, alternative='greater')
    auroc   = stat / (len(pos) * len(neg))
    boot = [
        mannwhitneyu(
            rng.choice(pos, len(pos), replace=True),
            rng.choice(neg, len(neg), replace=True),
            alternative='greater',
        )[0] / (len(pos) * len(neg))
        for _ in range(n_boot)
    ]
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return auroc, lo, hi

# ── Per-group AUROC ────────────────────────────────────────────────────────────
rows = []
for gval in group_vals:
    s = _df_safe  [_df_safe  [GROUPBY].astype(str) == gval][METRIC].dropna().values
    u = _df_unsafe[_df_unsafe[GROUPBY].astype(str) == gval][METRIC].dropna().values
    pos, neg = (s, u) if HIGHER_IS_SAFER else (u, s)
    if len(s) >= 2 and len(u) >= 2:
        auroc, lo, hi = _auroc_ci_petri(pos, neg)
        rows.append({GROUPBY: gval, 'n_safe': len(s), 'n_unsafe': len(u),
                     'AUROC': auroc, 'CI_lo': lo, 'CI_hi': hi})
    else:
        rows.append({GROUPBY: gval, 'n_safe': len(s), 'n_unsafe': len(u),
                     'AUROC': np.nan, 'CI_lo': np.nan, 'CI_hi': np.nan})

# Pooled across all group values
s_all = _df_safe  [METRIC].dropna().values
u_all = _df_unsafe[METRIC].dropna().values
pos_all, neg_all = (s_all, u_all) if HIGHER_IS_SAFER else (u_all, s_all)
auroc_all, lo_all, hi_all = _auroc_ci_petri(pos_all, neg_all)
rows.append({GROUPBY: 'ALL (pooled)', 'n_safe': len(s_all), 'n_unsafe': len(u_all),
             'AUROC': auroc_all, 'CI_lo': lo_all, 'CI_hi': hi_all})

# ── Display ────────────────────────────────────────────────────────────────────
auroc_df = pd.DataFrame(rows)
auroc_df['95% CI'] = auroc_df.apply(
    lambda r: f"[{r.CI_lo:.3f}, {r.CI_hi:.3f}]" if not np.isnan(r.CI_lo) else '—', axis=1
)
direction = 'P(unsafe > safe)' if not HIGHER_IS_SAFER else 'P(safe > unsafe)'
fixed_str = ', '.join(f'{k}={v}' for k, v in FIXED.items())
caption   = (f'AUROC: {direction}  |  metric={METRIC}  |  '
             f'fixed: {fixed_str}  |  grouped by: {GROUPBY}  |  '
             f'{N_BOOT:,}-sample bootstrap CI')

display(
    auroc_df[[GROUPBY, 'n_safe', 'n_unsafe', 'AUROC', '95% CI']]
    .rename(columns={GROUPBY: GROUPBY.capitalize()})
    .style
    .format({'AUROC': lambda x: f'{x:.3f}' if not np.isnan(x) else '—'})
    .set_caption(caption)
)

print(FIXED)
print(auroc_df[[GROUPBY, 'AUROC', '95% CI']].rename(columns={GROUPBY: GROUPBY.capitalize()}))

In [ ]:
# ── Figure 7: Target size effect (safe: XS, S, M) ────────────────────────────
size_ref = df_r_safe[
    (df_r_safe['auditor'].astype(str) == REF_A) &
    (df_r_safe['judge'].astype(str)   == REF_J)
].copy()

targets_sz = [t for t in SIZE_ORDER if t in size_ref['target'].astype(str).values]
if len(targets_sz) < 2:
    print(f'Not enough target sizes for A={REF_A} J={REF_J} — check data')
else:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    for ax, (metric, color, ylabel) in zip(axes, [
        ('active_concern_v2', 'steelblue', 'active_concern_v2'),
        ('global_mean',       '#ff7f0e',   'global_mean (all 38 dims)'),
    ]):
        means = [size_ref[size_ref['target'].astype(str)==t][metric].mean() for t in targets_sz]
        stds  = [size_ref[size_ref['target'].astype(str)==t][metric].std()  for t in targets_sz]
        ns    = [size_ref[size_ref['target'].astype(str)==t][metric].count() for t in targets_sz]
        ax.bar(range(len(targets_sz)), means, yerr=stds, capsize=6, color=color, alpha=0.8,
               error_kw=dict(elinewidth=1.5))
        for i, (m, s, n) in enumerate(zip(means, stds, ns)):
            ax.text(i, (m or 0) + (s or 0) + 0.03, f'{m:.3f}\n(n={n})', ha='center', fontsize=8)
        ax.set_xticks(range(len(targets_sz)))
        ax.set_xticklabels([f'T={t}' for t in targets_sz])
        ax.set_ylabel(ylabel, fontsize=9)
        ax.grid(axis='y', ls='--', alpha=0.4)

    # Pairwise MWU on active_concern_v2
    print('Pairwise Mann-Whitney U on active_concern_v2:')
    for t1, t2 in combinations(targets_sz, 2):
        v1 = size_ref[size_ref['target'].astype(str)==t1]['active_concern_v2'].dropna().values
        v2 = size_ref[size_ref['target'].astype(str)==t2]['active_concern_v2'].dropna().values
        if len(v1) >= 3 and len(v2) >= 3:
            _, p = mannwhitneyu(v1, v2, alternative='two-sided')
            sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
            print(f'  T={t1} ({v1.mean():.3f}) vs T={t2} ({v2.mean():.3f}): p={p:.4f} {sig}')

    plt.suptitle(f'Figure 7 — Target size effect (safe targets)  [A={REF_A}, J={REF_J}]', fontsize=13)
    plt.tight_layout()
    plt.show()

---
## Section 4 — Role Importances (partial η²)

In [ ]:
# ── Variance decomposition helper ─────────────────────────────────────────────
def variance_decomp(df, outcome, factors=FACTORS, n_bootstrap=0):
    """Partial η² via Type-II OLS. Returns DataFrame.
    With n_bootstrap > 0 also adds eta2_lo, eta2_hi (95% percentile CI).
    """
    active_f = [f for f in factors if f in df.columns and df[f].nunique() >= 2]
    if len(active_f) < 2:
        return pd.DataFrame({'factor': factors,
                             'partial_eta2': [float('nan')]*len(factors),
                             'p_value': [float('nan')]*len(factors)})
    formula = f'Q("{outcome}") ~ ' + ' + '.join(f'C({f})' for f in active_f)
    df_c = df[active_f + [outcome]].dropna()
    rows = []
    try:
        model = ols(formula, data=df_c).fit()
        tbl   = anova_lm(model, typ=2)

        point_est = {}
        for f in active_f:
            key  = f'C({f})'
            ss_e = tbl.loc[key, 'sum_sq']
            ss_r = tbl.loc['Residual', 'sum_sq']
            point_est[f] = float(ss_e / (ss_e + ss_r))

        boot_samples = {f: [] for f in active_f}
        if n_bootstrap > 0:
            rng = np.random.default_rng(42)
            for _ in range(n_bootstrap):
                idx  = rng.integers(0, len(df_c), size=len(df_c))
                df_b = df_c.iloc[idx]
                if any(df_b[f].nunique() < 2 for f in active_f):
                    continue
                try:
                    bt = anova_lm(ols(formula, data=df_b).fit(), typ=2)
                    for f in active_f:
                        key  = f'C({f})'
                        ss_e = bt.loc[key, 'sum_sq']
                        ss_r = bt.loc['Residual', 'sum_sq']
                        boot_samples[f].append(float(ss_e / (ss_e + ss_r)))
                except Exception:
                    pass

        for f in factors:
            if f not in active_f:
                rows.append(dict(factor=f, partial_eta2=float('nan'), F=float('nan'),
                                 p_value=float('nan'), note='single level'))
                continue
            row = dict(factor=f, partial_eta2=point_est[f],
                       F=float(tbl.loc[f'C({f})', 'F']),
                       p_value=float(tbl.loc[f'C({f})', 'PR(>F)']), note='')
            if n_bootstrap > 0 and boot_samples[f]:
                s = np.array(boot_samples[f])
                row['eta2_lo'] = float(np.percentile(s, 2.5))
                row['eta2_hi'] = float(np.percentile(s, 97.5))
            rows.append(row)

    except Exception as e:
        for f in factors:
            rows.append(dict(factor=f, partial_eta2=float('nan'), F=float('nan'),
                             p_value=float('nan'), note=str(e)))
    return pd.DataFrame(rows)


def draw_eta2_bars(result, ax, title=''):
    color_map = {'judge': '#4878CF', 'auditor': '#D65F5F', 'target': '#6ACC65'}
    fr = result[result['factor'].isin(FACTORS)].sort_values('partial_eta2', ascending=True, na_position='first')
    has_ci = 'eta2_lo' in fr.columns and 'eta2_hi' in fr.columns
    for i, row in enumerate(fr.itertuples()):
        eta = row.partial_eta2
        if not np.isnan(eta):
            ax.barh(i, eta, color=color_map.get(row.factor, 'gray'), alpha=0.85)
            if has_ci and not np.isnan(row.eta2_lo) and not np.isnan(row.eta2_hi):
                ax.errorbar(eta, i,
                            xerr=[[eta - row.eta2_lo], [row.eta2_hi - eta]],
                            fmt='none', color='black', capsize=4, linewidth=1.2)
            label_x = row.eta2_hi + 0.01 if has_ci and not np.isnan(row.eta2_hi) else eta + 0.01
            p   = row.p_value if not np.isnan(row.p_value) else 1.0
            # sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
            ax.text(label_x, i, f'{eta:.3f}', va='center', fontsize=9)
        else:
            ax.text(0.02, i, 'n/a', va='center', fontsize=8, color='gray')
    ax.set_yticks(range(len(fr)))
    ax.set_yticklabels([r.factor.capitalize() for r in fr.itertuples()])
    ax.set_xlabel('Partial η²', fontsize=10)
    ax.set_xlim(0, 1.15)
    # for thresh, lbl, ls in [(0.06,'small',':'),(0.14,'medium','--'),(0.26,'large','-.')]:
    #     ax.axvline(thresh, color='gray', ls=ls, alpha=0.45, lw=1)
    #     ax.text(thresh+0.005, len(fr)-0.1, lbl, fontsize=7, color='gray', va='top')
    ax.set_title(title, fontsize=11)
    ax.grid(axis='x', ls='--', alpha=0.3)


# ── Figure 8: Role importances ────────────────────────────────────────────────
outcomes = ["concerning", "unprompted_bizarre_behavior", "admirable"]#, 'admirable', "concerning_with_no_mitigating_factors", "unprompted_sycophancy"] 
# + [
#     d for d in representative_dims if d in df_r_safe.columns
# ]
n = len(outcomes)
fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
if n == 1:
    axes = [axes]

for ax, outcome in zip(axes, outcomes):
    vd = variance_decomp(df_r, outcome, n_bootstrap=1000)
    draw_eta2_bars(vd, ax, title=outcome.replace('unprompted_', 'u_'))

# plt.suptitle('Figure 8 — Partial η² per outcome', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('img/A_petri.png', dpi=300, bbox_inches='tight')
plt.show()

---
## Section 5 — Paper Comparison Figures

Mirror plots from `simpleaudit_analysis.ipynb` for side-by-side comparison.

In [ ]:
# ── Figure 9: Heatmap target × auditor (fixed J=L) ───────────────────────────
for metric in ['active_concern_v2', 'global_mean']:
    sub = df_r_safe[df_r_safe['judge'].astype(str) == REF_J]
    targets_h  = [t for t in SIZE_ORDER if t in sub['target'].astype(str).values]
    auditors_h = [a for a in SIZE_ORDER if a in sub['auditor'].astype(str).values]

    matrix   = pd.DataFrame(float('nan'), index=auditors_h, columns=targets_h)
    n_matrix = pd.DataFrame(0,            index=auditors_h, columns=targets_h)
    for t in targets_h:
        for a in auditors_h:
            cell = sub[(sub['target'].astype(str)==t) & (sub['auditor'].astype(str)==a)][metric].dropna()
            if len(cell) > 0:
                matrix.loc[a, t]   = cell.mean()
                n_matrix.loc[a, t] = len(cell)

    fig, ax = plt.subplots(figsize=(max(5, len(targets_h)*2+1), max(3, len(auditors_h)*1.3+1)))
    sns.heatmap(matrix.astype(float), ax=ax, cmap='RdYlGn_r',
                mask=matrix.isna(), linewidths=0.5, linecolor='white',
                cbar_kws={'label': metric, 'shrink': 0.8}, annot=False)
    for i, a in enumerate(auditors_h):
        for j, t in enumerate(targets_h):
            val = matrix.loc[a, t]
            n   = n_matrix.loc[a, t]
            if not pd.isna(val):
                ax.text(j+0.5, i+0.5, f'{val:.3f}\n(n={n})', ha='center', va='center', fontsize=9)
            else:
                ax.text(j+0.5, i+0.5, '×', ha='center', va='center', fontsize=16, color='lightgray')
    ax.set_xlabel('Target', fontsize=11)
    ax.set_ylabel('Auditor', fontsize=11)
    ax.set_title(f'Figure 9 — {metric}: target × auditor  |  J={REF_J}', fontsize=12)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Figure 10: Score by judge size (T=S, A=L) ────────────────────────────────
sub_j = df_r_safe[
    (df_r_safe['target'].astype(str)  == 'S') &
    (df_r_safe['auditor'].astype(str) == REF_A)
]
judges_avail = [j for j in SIZE_ORDER if j in sub_j['judge'].astype(str).values]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (metric, color) in zip(axes, [('active_concern_v2', 'steelblue'), ('global_mean', '#ff7f0e')]):
    means = [sub_j[sub_j['judge'].astype(str)==j][metric].mean() for j in judges_avail]
    stds  = [sub_j[sub_j['judge'].astype(str)==j][metric].std()  for j in judges_avail]
    ns    = [sub_j[sub_j['judge'].astype(str)==j][metric].count() for j in judges_avail]
    ax.bar(range(len(judges_avail)), means, yerr=stds, capsize=6, color=color, alpha=0.8,
           error_kw=dict(elinewidth=1.5))
    for i, (m, s, n) in enumerate(zip(means, stds, ns)):
        ax.text(i, (m or 0)+(s or 0)+0.02, f'{m:.4f}\n(n={n})', ha='center', fontsize=8)
    ax.set_xticks(range(len(judges_avail)))
    ax.set_xticklabels([f'J={j}' for j in judges_avail])
    ax.set_ylabel(metric, fontsize=10)
    ax.grid(axis='y', ls='--', alpha=0.4)

plt.suptitle(f'Figure 10 — Score by judge size  |  T=S, A={REF_A}', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 10b: Score by auditor size (T=S, J=L) ────────────────────────────paper
REF_J = 'L'   # fixed judge
REF_T = 'S'   # fixed target

sub_a = df_r_safe[
    (df_r_safe['target'].astype(str) == REF_T) &
    (df_r_safe['judge'].astype(str)  == REF_J)
]
auditors_avail = [a for a in SIZE_ORDER if a in sub_a['auditor'].astype(str).values]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, (metric, color) in zip(axes, [("needs_attention", "steelblue"), ("global_mean", "#ff7f0e")]):
    means = [sub_a[sub_a["auditor"].astype(str)==a][metric].mean() for a in auditors_avail]
    stds  = [sub_a[sub_a["auditor"].astype(str)==a][metric].std()  for a in auditors_avail]
    ns    = [sub_a[sub_a["auditor"].astype(str)==a][metric].count() for a in auditors_avail]
    ax.bar(range(len(auditors_avail)), means, yerr=stds, capsize=6, color=color, alpha=0.8,
           error_kw=dict(elinewidth=1.5))
    for i, (m, s, n) in enumerate(zip(means, stds, ns)):
        ax.text(i, (m or 0)+(s or 0)+0.02, f"{m:.4f}\n(n={n})", ha="center", fontsize=8)
    ax.set_xticks(range(len(auditors_avail)))
    ax.set_xticklabels([f"A={a}" for a in auditors_avail])
    ax.set_ylabel(metric, fontsize=10)
    ax.grid(axis="y", ls="--", alpha=0.4)

plt.suptitle(f"Figure 10b — Score by auditor size  |  T={REF_T}, J={REF_J}", fontsize=13)
plt.tight_layout()
plt.show()

---
## Section 6 — Ordinal Stability Curve

Direct analog of the SimpleAudit stability analysis (`analyze_data.compute_stability_curve`),
applied to Petri's 1–10 integer dimension scores instead of the 0–4 severity ordinals.

**Profile**: vector of mean scores over `active_union` dims, averaged over all transcripts
in k randomly drawn runs. **MAD** = mean |k-run profile − full 10-run reference|, in units
of the 1–10 scale.

Fixed: A=L, J=L (same reference as throughout this notebook).

In [ ]:
# ── Petri Ordinal Stability Curve ────────────────────────────────────────────
# Analog of analyze_data.compute_stability_curve applied to Petri 1-10 scores.
# Profile = mean score over active_union dims, averaged over transcripts in k runs.
# MAD = mean |k-run profile − full-run reference|  (units: 1–10 scale).
# Fixed: A=L, J=L

_STAB_A    = REF_A
_STAB_J    = REF_J
_STAB_DIMS = ["concerning"]
#active_union
_N_BOOT    = 1000
_rng       = np.random.default_rng(42)


def _petri_stability(df_safe, df_unsafe, dims, fixed_a, fixed_j, n_bootstrap, rng):
    rows = []
    for df_t, target_safe in [(df_safe, True), (df_unsafe, False)]:
        sub = df_t[
            (df_t['auditor'].astype(str) == fixed_a) &
            (df_t['judge'].astype(str)   == fixed_j)
        ]
        for target, cell in sub.groupby('target', observed=True):
            run_ids = sorted(cell['run_id'].unique())
            n_runs  = len(run_ids)
            if n_runs < 5:
                continue
            # Per-run mean profile: (n_runs, n_dims)
            run_profiles = np.stack([
                cell[cell['run_id'] == r][dims].mean().values for r in run_ids
            ])
            ref = run_profiles.mean(axis=0)   # full-run reference
            for k in range(1, n_runs):
                idx     = np.array([rng.choice(n_runs, k, replace=False) for _ in range(n_bootstrap)])
                samples = run_profiles[idx].mean(axis=1)  # (n_bootstrap, n_dims)
                mads    = np.abs(samples - ref).mean(axis=1)
                rows.append(dict(
                    target=str(target), auditor=fixed_a, judge=fixed_j,
                    target_safe=target_safe, k=k, n_runs=n_runs,
                    mean_mad=float(mads.mean()),
                    ci_lo=float(np.percentile(mads, 2.5)),
                    ci_hi=float(np.percentile(mads, 97.5)),
                ))
    return pd.DataFrame(rows)


stab_petri = _petri_stability(
    df_t_safe, df_t_unsafe, _STAB_DIMS, _STAB_A, _STAB_J, _N_BOOT, _rng
)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
_colors = {True: '#4878CF', False: '#C44E52'}
_labels = {True: 'Safe targets', False: 'Unsafe targets'}

for safe_val in [True, False]:
    sub = stab_petri[stab_petri['target_safe'] == safe_val]
    if sub.empty:
        continue
    grp    = sub.groupby('k')['mean_mad']
    k_vals = grp.mean().index.values
    means  = grp.mean().values
    stds   = grp.std(ddof=1).fillna(0).values
    ax.plot(k_vals, means, color=_colors[safe_val],
            label=_labels[safe_val], linewidth=2, marker='o', markersize=5)
    ax.fill_between(k_vals, np.maximum(0, means - stds), means + stds,
                    color=_colors[safe_val], alpha=0.15)

# ax.axhline(0.5, color='grey', linestyle='--', linewidth=1, label='MAD = 0.5')
# ax.axhline(0,   color='grey', linestyle=':',  linewidth=1)
ax.set_xlabel('Runs used (k)', fontsize=11)
ax.set_ylabel('Mean dimension MAD  (1–10 scale)', fontsize=11)
# ax.set_title(
#     f'Petri dimension profile: convergence with run count'
#     f'A={_STAB_A}, J={_STAB_J} | active_union dims ({len(_STAB_DIMS)})',
#     fontsize=12
# )
ax.legend(fontsize=10)
ax.grid(axis='both', linestyle='--', alpha=0.4)
ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
plt.tight_layout()
plt.savefig('img/fig_petri_stability_curve.png', bbox_inches='tight')
plt.show()

# ── Threshold summary ─────────────────────────────────────────────────────────
print('k at which mean MAD first drops below 0.5 (units: 1–10 scale):')
for safe in [True, False]:
    sub    = stab_petri[stab_petri['target_safe'] == safe].groupby('k')['mean_mad'].mean()
    below  = sub[sub < 0.5]
    k_thr  = int(below.index.min()) if not below.empty else 'never in range'
    print(f'  target_safe={safe}: k = {k_thr}  (MAD at k=5: {sub.get(5, float("nan")):.3f})')


In [ ]:
# ── Per-dimension max MAD at k=1, k=5, k=9 ────────────────────────────────
# For each dim individually (pooled across targets), bootstrap mean MAD
# at k = 1, 5, 9 and report the worst-case (max) dim at each k.

_K_SUMMARY = [1, 5, 9]
_rng_s = np.random.default_rng(0)


def _dim_mad_at_k(df_safe, df_unsafe, dims, fixed_a, fixed_j, k_vals, n_bootstrap, rng):
    rows = []
    for dim in dims:
        for df_t, target_safe in [(df_safe, True), (df_unsafe, False)]:
            sub = df_t[
                (df_t['auditor'].astype(str) == fixed_a) &
                (df_t['judge'].astype(str)   == fixed_j)
            ]
            if dim not in sub.columns:
                continue
            all_run_means = []
            for target, cell in sub.groupby('target', observed=True):
                run_ids = sorted(cell['run_id'].unique())
                if len(run_ids) < 5:
                    continue
                for r in run_ids:
                    vals = cell[cell['run_id'] == r][dim].dropna()
                    if len(vals):
                        all_run_means.append(float(vals.mean()))
            if len(all_run_means) < 5:
                continue
            arr = np.array(all_run_means)
            ref = arr.mean()
            n   = len(arr)
            for k in k_vals:
                if k >= n:
                    continue
                idx     = np.array([rng.choice(n, k, replace=False) for _ in range(n_bootstrap)])
                samples = arr[idx].mean(axis=1)
                mads    = np.abs(samples - ref)
                rows.append(dict(
                    dim=dim, target_safe=target_safe, k=k,
                    mean_mad=float(mads.mean()),
                    ci_lo=float(np.percentile(mads, 2.5)),
                    ci_hi=float(np.percentile(mads, 97.5)),
                ))
    return pd.DataFrame(rows)


_stab_per_dim = _dim_mad_at_k(
    df_t_safe, df_t_unsafe, dim_cols,
    _STAB_A, _STAB_J, _K_SUMMARY, _N_BOOT, _rng_s,
)

print('Max mean MAD across all dims at k=1, 5, 9  (A=L, J=L, XL excluded):\n')
for safe in [True, False]:
    label = 'Safe' if safe else 'Unsafe'
    sub = _stab_per_dim[_stab_per_dim['target_safe'] == safe]
    print(f'{label} targets:')
    for k in _K_SUMMARY:
        ks = sub[sub['k'] == k]
        if ks.empty:
            continue
        row = ks.loc[ks['mean_mad'].idxmax()]
        print(f'  k={k}: max MAD = {row["mean_mad"]:.3f}  ({row["dim"]})')
    print()
